In [1]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
import pdfplumber
import os
from google import genai
from dotenv import load_dotenv
import numpy as np


load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))


In [2]:
# pdf_path = r"D:\JEEVA\GENAI\Ai_with_Rag\6th_Social_Science_Term_I_EM - www.tntextbooks.in.pdf"
pdf_path = r"D:\JEEVA\GENAI\Ai_with_Rag\Group IV Syllabus.pdf"


In [3]:
# all_text = []
# with pdfplumber.open(pdf_path) as pdf:
#     for page_num, page in enumerate(pdf.pages[8:12], start=1):
#         text = page.extract_text() or ""
        
#         # all_text.append(f"\n--- Page {page_num} ---\n")
#         all_text.append(text)


# full_text = "\n".join(all_text)

with pdfplumber.open(pdf_path) as pdf:
    full_text = "\n".join(
        page.extract_text() or ""
        for page in pdf.pages[:]
    )
    

In [4]:
full_text

'Combined Civil Services Examination – IV (Group IV Services)\n[Single Paper in SSLC Standard consisting of General Studies (75 Questions), Aptitude and\nMental Ability (25 Questions) and Tamil Eligibility-cum-Scoring Test (100 Questions)]\nCode: 496\nPart A: General Studies (SSLC Standard – 75 Questions)\nUnit I: General Science (5 Questions)\nNature of Universe - Measurement of physical quantities - General scientific laws in motion - force,\npressure, and energy - Everyday application of the basic principles of mechanics, electricity,\nmagnetism, light, sound, heat, and nuclear physics in our daily life; Elements and compounds,\nacids, bases, salts, petroleum products, fertilizers, pesticides, metallurgy, and food adulterants;\nmain concepts of life science, classification of living organisms, evolution, genetics, physiology,\nnutrition, health and hygiene, human diseases; Environmental science; Latest inventions in\nscience and technology; Current affairs.\nUnit II: Geography (5 Qu

In [5]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
    separators=["\n\n","\n"," "]
)
text=text_splitter.split_text(full_text)

In [6]:
text
len(text)

28

In [7]:

def convert_embed(text):
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text
    )
    # print(result.embeddings[0].values[:10])
    return result.embeddings[0].values


In [8]:
chunk_embedding=[]

for txt in text[:]:
    chunk_embedding.append(convert_embed(text=txt))

In [ ]:
chunk_embedding[0]


[0.0005823033,
 -0.0075870026,
 0.011306163,
 -0.04240944,
 -0.0064483345,
 0.014691736,
 -0.010305229,
 0.021550084,
 -0.025734466,
 0.0005192612,
 -0.034660205,
 -0.0049048862,
 -0.0049433936,
 0.029467603,
 0.12051989,
 -0.0048285415,
 -0.020249728,
 -0.019980295,
 0.004430643,
 -0.009985945,
 0.009840796,
 -0.0068466505,
 0.021723825,
 -0.033587348,
 -0.026884802,
 0.014711414,
 0.036249723,
 0.023224311,
 0.023333196,
 -0.0048602917,
 -0.010080506,
 0.0017525088,
 0.0071883374,
 0.039584566,
 0.0057530967,
 0.011258338,
 0.00983762,
 -0.0061844825,
 0.010853072,
 0.0033438187,
 0.023770053,
 -0.00843151,
 0.021586362,
 0.00365473,
 -0.0069724736,
 0.004336334,
 0.02312659,
 -0.006424785,
 -0.011698603,
 0.03915156,
 -0.025653388,
 0.005352288,
 -0.012953941,
 -0.18654144,
 -0.02954974,
 -0.038819507,
 -0.027596312,
 -0.022180595,
 0.0019831143,
 0.00031664874,
 -0.0042154747,
 0.010637441,
 -0.013141951,
 -0.010331975,
 -0.016700463,
 -0.030259952,
 0.008168269,
 0.011111504,
 -0.

In [26]:
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))



In [27]:
vec1=convert_embed("worst")
vec2=convert_embed("bad")
cosine_similarity(vec1,vec2)

np.float64(0.747290596404167)

In [ ]:
import faiss
import numpy as np

documents = []

def create_faiss_index(dimension): #3072
    return faiss.IndexFlatL2(dimension)

def store_embeddings(index, embeddings, docs):
    global documents

    vectors = np.array(embeddings, dtype=np.float32)
    index.add(vectors)

    documents.extend(docs)


def retrieve_documents(index, query_embedding, top_k=3):
    query_vector = np.array([query_embedding], dtype=np.float32)

    distances, indices = index.search(query_vector, top_k)

    retrieved_docs = []

    for idx in indices[0]:
        if idx != -1:
            retrieved_docs.append(documents[idx])

    return retrieved_docs

In [29]:
dimensions=len(chunk_embedding[0])
idx=create_faiss_index(dimensions)
store=store_embeddings(idx,chunk_embedding,text[:])


In [30]:
dimensions

3072

In [31]:
idx

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x00000203DE4E9970> >

In [32]:
store

In [33]:
text[0]

'Combined Civil Services Examination – IV (Group IV Services)\n[Single Paper in SSLC Standard consisting of General Studies (75 Questions), Aptitude and\nMental Ability (25 Questions) and Tamil Eligibility-cum-Scoring Test (100 Questions)]\nCode: 496\nPart A: General Studies (SSLC Standard – 75 Questions)\nUnit I: General Science (5 Questions)\nNature of Universe - Measurement of physical quantities - General scientific laws in motion - force,\npressure, and energy - Everyday application of the basic principles of mechanics, electricity,\nmagnetism, light, sound, heat, and nuclear physics in our daily life; Elements and compounds,\nacids, bases, salts, petroleum products, fertilizers, pesticides, metallurgy, and food adulterants;\nmain concepts of life science, classification of living organisms, evolution, genetics, physiology,\nnutrition, health and hygiene, human diseases; Environmental science; Latest inventions in\nscience and technology; Current affairs.\nUnit II: Geography (5 Qu

In [34]:
query="Translation"
em_query=convert_embed(query)
ret=retrieve_documents(idx,em_query,1)


In [35]:
ret

['basics).\nUnit V: Reading Comprehension (20 Questions)\nUnseen passages (News Paper, Headlines, Editorials, Government related News), Question Types –\nStrong question, Weak question, Match the following, Sentence Completion, Ascertainment of facts\n(Multiple Choice Question) - Choose the best response.\nUnit VI: Translation (5 Questions)\nWord Translation, Sentence Translation, Tense related translation tasks, Tense / Voice related tasks.\nUnit VII: Literary Works (15 Questions) (SSLC Standard)\nFigures of Speech; Appreciation and Analysis of Poetry; Lines of Significance\nPoems: I Dream of Spices – Raj Arumugam; The Crocodile – Lewis Carroll; Teamwork – Edger Albert\nGuest; From a Railway Carriage – Robert Louis Balfour Stevenson; A Tragic Story – William Makepeace\nThackeray; Sea Fever – John Masefield; Courage – Edger Albert Guest; The Age of Chivalry – George\nKrokos; Wandering Singers – Sarojini Naidu; The Listeners – Walter de la mere; Your Space – David\nBates; Special Hero –